# Clase 20 · Texto sucio con regex, funciones propias y loops
**Ciencia de Datos para Economía y Negocios - FCE-UBA**

Notebook para ir corriendo en clase, celda por celda (en RStudio, Ctrl+Shift+Enter dentro de cada chunk; en Jupyter o Colab, Shift+Enter). La pregunta que ordena la clase: **¿cómo se convierte texto libre en variables que se puedan analizar, sin escribir el mismo código veinte veces?**

1. **Texto y expresiones regulares**: `str_detect()`, `str_replace()` y `str_extract()` para limpiar texto sucio y sacarle datos. Lo hacemos con 44 mil avisos de departamentos en venta de Properati, escritos a mano por las inmobiliarias.
2. **Funciones propias**: cuando hay que repetir la misma operación muchas veces, se escribe una vez y se usa siempre.
3. **Loops y condicionales**: `for`, `if`/`else`, `next`, y cuándo conviene no usar un loop.
4. **Integrador**: de la descripción del aviso a una regresión del precio.

La primera vez hace falta internet: los datos se bajan y quedan guardados en `datos_clase20/`.

**En Google Colab** (versión `.ipynb`): Entorno de ejecución → Cambiar tipo de entorno de ejecución → **R**.

In [1]:
# Paquetes: instala solo los que faltan
paquetes <- c("tidyverse", "stringi", "modelsummary")
faltan <- setdiff(paquetes, rownames(installed.packages()))
if (length(faltan) > 0) install.packages(faltan)

suppressWarnings(suppressPackageStartupMessages({
  library(tidyverse)      # incluye stringr: todas las funciones str_*()
  library(modelsummary)
}))

options(repr.plot.width = 10, repr.plot.height = 5, repr.plot.res = 110)   # tamaño de gráficos en Jupyter
theme_set(theme_minimal(base_size = 12))

# Muestra una tabla de modelsummary. En RStudio es la tabla de siempre; en un
# notebook de Jupyter o Colab la convierte en tabla markdown para que se vea bien.
mostrar_tabla <- function(modelos, ..., notas = NULL) {
  if (!isTRUE(getOption("jupyter.in_kernel"))) {
    return(modelsummary(modelos, ..., notes = notas))
  }
  df <- modelsummary(modelos, ..., output = "data.frame")
  df$term[df$statistic == "std.error"] <- ""
  df <- df[, c("term", names(modelos))]
  names(df)[1] <- " "
  tabla <- paste(knitr::kable(df, format = "pipe",
                              align = c("l", rep("r", length(modelos)))),
                 collapse = "\n")
  notas <- gsub("*", "\\*", notas, fixed = TRUE)
  IRdisplay::display_markdown(paste0(tabla, "\n\n", paste(notas, collapse = "  \n")))
}

### Datos

Avisos de departamentos en venta en CABA publicados en Properati (2018-2019), con el título y la descripción tal como los escribió cada inmobiliaria. La función `con_cache()` baja el archivo la primera vez (unos 65 MB), se queda con los departamentos en venta y guarda una versión liviana en `datos_clase20/`. Las veces siguientes la lee de ahí.

In [2]:
options(timeout = 900)

carpeta_datos <- "datos_clase20"
dir.create(carpeta_datos, showWarnings = FALSE)

con_cache <- function(nombre, crear) {
  archivo <- file.path(carpeta_datos, paste0(nombre, ".rds"))
  if (!file.exists(archivo)) saveRDS(crear(), archivo)
  readRDS(archivo)
}

bajar_avisos <- function() {
  url <- "https://raw.githubusercontent.com/martintinch0/CienciaDeDatosParaCuriosos/master/data/datosProperati.csv"
  tmp <- tempfile(fileext = ".csv")
  download.file(url, tmp, mode = "wb")
  avisos <- read_delim(tmp, delim = ";", locale = locale(encoding = "latin1"),
                       show_col_types = FALSE) |>
    filter(property_type == "Departamento", operation_type == "Venta") |>
    transmute(id          = row_number(),
              fecha       = as.Date(created_on),
              barrio      = BARRIO,
              ambientes   = rooms,
              m2          = surface_covered,
              m2_total    = surface_total,
              precio      = price,             # en dólares
              titulo      = title,
              descripcion = description)
  unlink(tmp)
  avisos
}

avisos <- con_cache("avisos_deptos", bajar_avisos)
glimpse(avisos)

Rows: 43,996
Columns: 9
$ id          <int> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17,…
$ fecha       <date> 2019-02-23, 2019-02-23, 2019-02-23, 2019-02-23, 2019-02-2…
$ barrio      <chr> "BALVANERA", "BOEDO", "BOEDO", "BOEDO", "BOEDO", "BOEDO", …
$ ambientes   <dbl> 1, 2, 1, 1, 1, 1, 2, NA, 2, 3, NA, 1, 2, 4, 2, 2, 2, 1, 3,…
$ m2          <dbl> 34, 64, 46, 51, 46, 50, 64, 26, 57, 52, 35, 32, 38, 96, 39…
$ m2_total    <dbl> 37, 70, 77, 56, 49, 78, 70, 26, 108, 56, 35, 32, 38, 103, …
$ precio      <dbl> 68000, 130215, 120108, 110000, 92153, 130000, 130215, 7500…
$ titulo      <chr> "Juan J Castelli  100 12°  - U$D 68.000 - Departamento en …
$ descripcion <chr> "Departemento de 37 m2 total con Balcon Piso 12 muy lumino…


# Parte 1. Texto y expresiones regulares

## 1.1 Cómo viene el texto

Tres descripciones al azar (recortadas a 400 caracteres):

In [3]:
set.seed(19)
avisos |> slice_sample(n = 3) |> pull(descripcion) |> str_trunc(400) |> writeLines(sep = "\n\n")

OPORTUNIDAD 40 MTS PRECIO INVERSOR Se deja constancia que las caracteristicas, medidas, expensas y superficies  consignadas son aproximadas, sujetos a verificación y/o ajuste. El precio del inmueble puede ser modificado sin previo aviso. Fotos de carácter no contractual. Las unidades publicadas están sujetas a disponibilidad.

Con excelente ubicación en el barrio de Recoleta, a dos cuadras de Av. Santa Fe y a dos cuadras del Shopping Alto Palermo les ofrecemos en venta este comodo departamento de dos ambientes totalmente amoblado.Segundo piso a la calle super luminoso. Cocina semi integrada con espacio para  lavavajillas o lavarropas, toilette de recepción, living comedor con salida a un gran balcón corrido, el mismo...

Xintel (BRI-BR1-140047) DEPTO TIPO CASA Y PISO 4 AMBIENTES. FRENTE. 76 M2 TOTALES. TODO A NUEVO. 2 BALCONES CON PROTECCION. APTO PROFESIONAL. DOBLE ENTRADA. BAÑO COMPLETO + TOILETTE DE RECEPCION (O LAVADERO). COCINA INTEGRADA. 4 PLACARDS. SERVICIOS INDIVIDUALES. PISOS 

Y algunos títulos:

In [4]:
set.seed(4)
avisos |> slice_sample(n = 12) |> pull(titulo)

[1] "DEPARTAMENTO 2 AMBIENTES APTO CREDITO/PROFESIONAL VENTA ALMAGRO"                                               
 [2] "DEPARTAMENTO EN VENTA"                                                                                         
 [3] "Departamento de 2 ambientes en Venta en Balvanera"                                                             
 [4] "Departamento en Venta"                                                                                         
 [5] "Departamento de 1 ambiente en Venta en Palermo"                                                                
 [6] "DEPARTAMENTO EN VENTA"                                                                                         
 [7] "Departamento - Nuñez"                                                                                          
 [8] "Departamento - Belgrano Barrancas"                                                                             
 [9] "A estrenar, monoambiente en venta Be Libertador. Oportunidad retasado"                                         
[10] "Venta. Departamento 2 ambientes. Balcón. Cochera. Apto profesional. Cochera. Balcón. Terrero y Yerbal. Flores."
[11] "Monoambiente con Balcon y Cocina separada Super Luminoso UNICO"                                                
[12] "DEPARTAMENTO EN VENTA"

La misma información aparece escrita de mil formas. Cuántos departamentos traen la cantidad de ambientes y los metros cargados en las columnas, y cuántos no:

In [ ]:
avisos |> summarise(avisos = n(),
                    sin_ambientes = sum(is.na(ambientes)),
                    sin_m2        = sum(is.na(m2)))

| Suciedad | Ejemplos en los avisos |
|:--|:--|
| Mayúsculas y minúsculas | `DEPARTAMENTO`, `Departamento`, `departamento` |
| Espacios de más | `"Venta  semipiso Categoria en  Belgrano"` |
| Tildes que a veces están y a veces no | `Núñez` y `Nunez`, `balcón` y `balcon` |
| Restos de HTML | `<br>`, `<p>`, `&nbsp;` |
| Abreviaturas y sinónimos | `3 amb`, `3amb`, `3 ambientes`, `tres ambientes` |
| Unidades escritas distinto | `56 m2`, `56m²`, `56 mts2`, `56 metros cuadrados` |
| Números con formato | `$ 3.700`, `u$s 120.000`, `36,20 m2` |

La idea de esta parte: **primero normalizar** (que el mismo dato se escriba igual), **después buscar** patrones y **sacar** de ahí los datos.

## 1.2 Las funciones de `stringr`

Todas empiezan con `str_`, reciben primero el texto (así se usan con `|>` y dentro de `mutate()`) y después el patrón.

| Función | Qué hace | Devuelve |
|:--|:--|:--|
| `str_detect(x, patron)` | ¿Aparece el patrón? | `TRUE`/`FALSE` |
| `str_count(x, patron)` | Cuántas veces aparece | Número |
| `str_replace(x, patron, nuevo)` | Reemplaza la **primera** aparición | Texto |
| `str_replace_all(x, patron, nuevo)` | Reemplaza **todas** | Texto |
| `str_remove()`, `str_remove_all()` | Reemplazan por nada | Texto |
| `str_extract(x, patron)` | Saca la primera aparición (o `NA`) | Texto |
| `str_extract_all(x, patron)` | Saca todas | Lista de vectores |
| `str_match(x, patron)` | Como `str_extract()`, pero separa los grupos `( )` | Matriz |
| `str_to_lower()`, `str_to_upper()`, `str_to_title()` | Cambian mayúsculas y minúsculas | Texto |
| `str_squish()` | Saca espacios al principio y al final, y deja uno solo entre palabras | Texto |
| `str_trunc()`, `str_sub()`, `str_length()` | Recortar y medir | Texto o número |
| `str_view(x, patron)` | Muestra qué parte del texto coincide con el patrón | Para mirar |

## 1.3 Arreglar texto: normalizar antes de comparar

Un ejemplo de pizarrón: la columna "provincia" de una encuesta online donde cada persona escribió lo que quiso.

In [5]:
encuesta <- tibble(
  provincia = c("Buenos Aires", " buenos aires ", "Bs. As.", "BSAS", "Pcia. de Buenos Aires",
                "CABA", "Capital Federal", "Ciudad Autónoma de Buenos Aires", "C.A.B.A.",
                "Córdoba", "cordoba", "CÓRDOBA  ", "Santa Fé", "santa fe", "Sta. Fe"),
  sueldo    = c("$ 850.000", "850000", "$850.000,00", "U$S 900", "1.200.000",
                "$ 1.500.000", "1500000", "USD 1.100", "$ 2.000.000,50",
                "$ 700.000", "700 mil", "$650.000", "$ 900.000", "900.000", "$ 1.000.000")
)
encuesta |> count(provincia)

provincia,n
<chr>,<int>
buenos aires,1
BSAS,1
Bs. As.,1
Buenos Aires,1
C.A.B.A.,1
CABA,1
Capital Federal,1
Ciudad Autónoma de Buenos Aires,1
CÓRDOBA,1


Para R son 15 provincias distintas. Primero, lo que se arregla con funciones (sin regex): espacios, mayúsculas y tildes.

In [6]:
encuesta <- encuesta |>
  mutate(prov = provincia |>
           str_squish() |>                                # espacios
           str_to_lower() |>                              # mayúsculas
           stringi::stri_trans_general("Latin-ASCII"))    # tildes: "córdoba" -> "cordoba"
encuesta |> count(prov)

prov,n
<chr>,<int>
bs. as.,1
bsas,1
buenos aires,2
c.a.b.a.,1
caba,1
capital federal,1
ciudad autonoma de buenos aires,1
cordoba,3
pcia. de buenos aires,1


Quedan 11. Lo que sigue son **abreviaturas y sinónimos**: ninguna función genérica sabe que "bs. as." es Buenos Aires. Para eso, una regla por categoría con `case_when()` y `str_detect()`:

In [7]:
encuesta <- encuesta |>
  mutate(prov = case_when(
    str_detect(prov, "caba|c\\.a\\.b\\.a|capital|ciudad") ~ "CABA",
    str_detect(prov, "buenos aires|bs\\.? ?as") ~ "Buenos Aires",
    str_detect(prov, "cordoba") ~ "Córdoba",
    str_detect(prov, "santa fe|sta\\.? fe") ~ "Santa Fe",
    .default = NA_character_))
encuesta |> count(prov)

prov,n
<chr>,<int>
Buenos Aires,5
CABA,4
Córdoba,3
Santa Fe,3


> 💡 **Lectura.** De 15 escrituras distintas a 4 provincias. El orden de las reglas importa: "Ciudad Autónoma de Buenos Aires" contiene "buenos aires", así que la regla de CABA tiene que ir **antes** que la de Buenos Aires. `case_when()` se queda con la primera regla que se cumple.

> ⚠️ **Atención.** Para sacar tildes, `stringi::stri_trans_general(x, "Latin-ASCII")` funciona igual en Windows, Mac y Colab. La alternativa que suele aparecer en internet, `iconv(x, to = "ASCII//TRANSLIT")`, da resultados distintos según el sistema operativo (en Windows a veces deja `"c?rdoba"`).

Con los números pasa lo mismo: `as.numeric()` no entiende `"$ 850.000"`. `parse_number()` (de `readr`) descarta lo que no es número, pero hay que decirle que en Argentina el punto separa miles y la coma, decimales:

In [8]:
encuesta |>
  transmute(sueldo,
            as_numeric   = suppressWarnings(as.numeric(sueldo)),
            parse_number = parse_number(sueldo, locale = locale(grouping_mark = ".", decimal_mark = ",")),
            en_dolares   = str_detect(sueldo, regex("u\\$s|usd", ignore_case = TRUE)))

sueldo,as_numeric,parse_number,en_dolares
<chr>,<dbl>,<dbl>,<lgl>
$ 850.000,NA,850000,FALSE
850000,850000,850000,FALSE
"$850.000,00",NA,850000,FALSE
U$S 900,NA,900,TRUE
1.200.000,NA,1200000,FALSE
$ 1.500.000,NA,1500000,FALSE
1500000,1500000,1500000,FALSE
USD 1.100,NA,1100,TRUE
"$ 2.000.000,50",NA,2000001,FALSE


> 💡 **Lectura.** `as.numeric()` entiende solo tres de los 15 valores, y uno lo lee **mal**: "900.000" queda como 900, porque toma el punto como separador decimal, y no avisa. `parse_number()` entiende casi todo, pero no sabe que "700 mil" son 700.000 (devuelve 700) ni que dos sueldos están en dólares: para eso hace falta mirar el texto con `str_detect()`. Un número limpio no siempre es un número correcto.

## 1.4 El lenguaje de las expresiones regulares

Una **expresión regular** (regex) es un patrón que describe un conjunto de textos. `"cochera"` busca literalmente esa palabra; `"\\d+ ?amb"` busca "uno o más dígitos, un espacio opcional y amb". Los símbolos que más se usan:

| Símbolo | Qué significa | Ejemplo | Coincide con |
|:--|:--|:--|:--|
| `.` | Cualquier caracter | `"m.s"` | "mts", "mas", "m s" |
| `\\d` | Un dígito (0-9) | `"\\d\\d"` | "42" |
| `\\w` | Una letra, dígito o `_` | `"\\w+"` | una palabra |
| `\\s` | Un espacio (o tab, salto de línea) | `"3\\samb"` | "3 amb" |
| `[abc]` | Uno de esos caracteres | `"[mM]2"` | "m2", "M2" |
| `[^abc]` | Cualquiera **menos** esos | `"[^0-9]"` | algo que no sea dígito |
| `?` | Lo anterior, 0 o 1 vez | `"ambientes?"` | "ambiente", "ambientes" |
| `+` | Lo anterior, 1 o más veces | `"\\d+"` | "3", "120", "1500" |
| `*` | Lo anterior, 0 o más veces | `"3 *amb"` | "3amb", "3 amb", "3   amb" |
| `{n,m}` | Lo anterior, entre n y m veces | `"\\d{1,2}"` | "3", "12" |
| `^`, `$` | Principio y final del texto | `"^venta"` | empieza con "venta" |
| `\\b` | Borde de palabra | `"\\bsum\\b"` | "sum", no "consumo" |
| `a\|b` | Una cosa o la otra | `"cochera\|garage"` | cualquiera de las dos |
| `( )` | Agrupa (y "captura" para `str_match()`) | `"(\\d+) amb"` | guarda el número |

Los caracteres con significado especial (`. $ ( ) [ ] ? + * ^ |`) se escriben con `\\` adelante cuando se los busca literalmente: `"\\$"` busca el signo pesos, `"\\."` busca un punto. En R la barra va **doble** porque el texto entre comillas ya usa `\` para otras cosas (como `"\n"`).

`str_view()` muestra entre `<>` qué parte del texto coincide. Es la mejor forma de probar un patrón antes de usarlo:

In [11]:
ejemplos <- c("Depto 3 amb. luminoso", "3 ambientes c/cochera", "2amb", "Monoambiente",
              "AMBIENTE AMPLIO", "a 300 mts del subte", "56 m2 totales", "Expensas $ 3.700")

str_view(ejemplos, "\\d")       # el primer dígito... y los siguientes, uno por uno
str_view(ejemplos, "\\d+")      # números enteros
str_view(ejemplos, "\\d+ ?amb") # número seguido de "amb", con o sin espacio

[1] │ Depto <3> amb. luminoso
[2] │ <3> ambientes c/cochera
[3] │ <2>amb
[6] │ a <3><0><0> mts del subte
[7] │ <5><6> m<2> totales
[8] │ Expensas $ <3>.<7><0><0>

[1] │ Depto <3> amb. luminoso
[2] │ <3> ambientes c/cochera
[3] │ <2>amb
[6] │ a <300> mts del subte
[7] │ <56> m<2> totales
[8] │ Expensas $ <3>.<700>

[1] │ Depto <3 amb>. luminoso
[2] │ <3 amb>ientes c/cochera
[3] │ <2amb>

In [12]:
str_view(ejemplos, "\\$ ?[\\d.]+")            # un monto en pesos
str_view(ejemplos, regex("^\\w+", ignore_case = TRUE))   # la primera palabra

[8] │ Expensas <$ 3.700>

[1] │ <Depto> 3 amb. luminoso
[2] │ <3> ambientes c/cochera
[3] │ <2amb>
[4] │ <Monoambiente>
[5] │ <AMBIENTE> AMPLIO
[6] │ <a> 300 mts del subte
[7] │ <56> m2 totales
[8] │ <Expensas> $ 3.700

> ⚠️ **Atención.** Las regex distinguen mayúsculas: `"amb"` no encuentra "AMBIENTE". Dos soluciones: pasar todo a minúsculas antes (lo que hacemos en 1.5) o escribir el patrón como `regex("amb", ignore_case = TRUE)`. Y si se busca un texto **literal** con muchos símbolos (por ejemplo `"u$s (USD)"`), `fixed("u$s (USD)")` evita tener que escapar cada uno.

## 1.5 `str_replace()`: limpiar

La descripción trae restos de HTML: etiquetas (`<br>`, `<p>`) y "entidades" (`&nbsp;`, `&amp;`). Un patrón para cada cosa:

- `"<[^>]+>"`: un `<`, uno o más caracteres que no sean `>`, y un `>`. Es decir, cualquier etiqueta.
- `"&[a-z]+;"`: un `&`, letras y un `;`.

In [13]:
ej <- avisos$descripcion[str_detect(avisos$descripcion, "<p>")][2] |> str_trunc(300)
ej
ej |> str_replace_all("<[^>]+>", " ") |> str_squish()

[1] "Xintel (CRY-CRY-87) Venta de Departamento 2 AMBIENTES en Palermo, Capital Federal<p><br></p><p>AMPLIO LIVING COMEDOR, TOTALMENTE RECICLADO A NUEVO, AL FRENTE, VISTA ABIERTA, LUMINOSO, A UNA CUADRA Y MEDIA DE AV. SANTA FE.</p> -  - SEBASTIAN CELERY BROKER INMOBILIARIO"

[1] "Xintel (CRY-CRY-87) Venta de Departamento 2 AMBIENTES en Palermo, Capital Federal AMPLIO LIVING COMEDOR, TOTALMENTE RECICLADO A NUEVO, AL FRENTE, VISTA ABIERTA, LUMINOSO, A UNA CUADRA Y MEDIA DE AV. SANTA FE. - - SEBASTIAN CELERY BROKER INMOBILIARIO"

`str_replace()` cambia solo la primera aparición; casi siempre se quiere `str_replace_all()`. Cuántos avisos tienen HTML:

In [ ]:
mean(str_detect(avisos$descripcion, "<[^>]+>"))

Ahora la limpieza completa, para la descripción y para el título. El orden es: sacar HTML, pasar a minúsculas, sacar tildes y por último los espacios (porque los pasos anteriores dejan espacios dobles).

In [14]:
avisos <- avisos |>
  mutate(desc = descripcion |>
           str_replace_all("<[^>]+>", " ") |>
           str_replace_all("&[a-z]+;", " ") |>
           str_to_lower() |>
           stringi::stri_trans_general("Latin-ASCII") |>
           str_squish())

avisos <- avisos |>
  mutate(tit = titulo |>
           str_replace_all("<[^>]+>", " ") |>
           str_replace_all("&[a-z]+;", " ") |>
           str_to_lower() |>
           stringi::stri_trans_general("Latin-ASCII") |>
           str_squish())

avisos |> select(titulo, tit) |> slice(1:5)

titulo,tit
<chr>,<chr>
Juan J Castelli 100 12° - U$D 68.000 - Departamento en Venta,juan j castelli 100 12° - u$d 68.000 - departamento en venta
Carlos Calvo 3700 4° - U$D 130.215 - Departamento en Venta,carlos calvo 3700 4° - u$d 130.215 - departamento en venta
Carlos Calvo 3700 1° - U$D 120.108 - Departamento en Venta,carlos calvo 3700 1° - u$d 120.108 - departamento en venta
Castro Barros 1100 4° - U$D 110.000 - Departamento en Venta,castro barros 1100 4° - u$d 110.000 - departamento en venta
Carlos Calvo 3700 4° - U$D 92.153 - Departamento en Venta,carlos calvo 3700 4° - u$d 92.153 - departamento en venta


> ⚠️ **Atención.** La columna original (`descripcion`, `titulo`) **no se pisa**: la versión limpia va en una columna nueva. Así, cuando un resultado parece raro, se puede volver a mirar qué decía el aviso.

### Reemplazos con grupos y varios reemplazos juntos

Entre paréntesis se "captura" una parte del patrón, y en el reemplazo se la recupera con `\\1`, `\\2`... Sirve para reordenar o separar: `"3amb"` → `"3 amb"`.

In [21]:
str_replace_all(c("3amb", "2amb c/cochera", "1 amb"), "(\\d)(amb)", "\\1 \\2")

[1] "3 amb"           "2 amb c/cochera" "1 amb"

Con un vector con nombres, `str_replace_all()` hace varios reemplazos de una vez (`patrón = reemplazo`). Por ejemplo, para unificar las unidades:

In [22]:
unidades <- c("m²" = "m2", "mts2" = "m2", "mt2" = "m2", "metros cuadrados" = "m2")
str_replace_all(c("56m²", "56 mts2", "56 metros cuadrados", "56 m2"), unidades)

[1] "56m2"  "56 m2" "56 m2" "56 m2"

## 1.6 `str_detect()`: crear variables a partir del texto

`str_detect()` devuelve `TRUE`/`FALSE`: es una **dummy** lista para usar.

In [26]:
avisos <- avisos |>
  mutate(cochera = str_detect(desc, "cochera|garage|guardacoche"))
mean(avisos$cochera)

[1] 0.3420538

Antes de confiar, **mirar el contexto** de lo que se encontró. Con `".{0,20}"` antes y después del patrón se ven 20 caracteres alrededor:

In [ ]:
set.seed(1)
avisos |> filter(cochera) |> slice_sample(n = 8) |>
  pull(desc) |> str_extract(".{0,25}(cochera|garage|guardacoche).{0,25}")

Las palabras que más aparecen justo antes de "cochera":

In [ ]:
avisos$desc |> str_extract("\\w+ cochera") |> table() |> sort(decreasing = TRUE) |> head(20)

> 💡 **Lectura.** La mayoría de las menciones son positivas ("con cochera", "y cochera", "incluye cochera"), pero aparecen "opcional cochera", "alquilar cochera", "no cochera" y "sin cochera": el aviso menciona la cochera para decir que **no** viene con el departamento. Una dummy que solo busca la palabra los cuenta como si la tuvieran.

Se corrige con un segundo patrón para las negaciones:

In [ ]:
negacion <- "(sin|no|opcional|opcion|alquilar|alquiler de)( de)? (cochera|garage)"
avisos <- avisos |>
  mutate(cochera = str_detect(desc, "cochera|garage|guardacoche") &
                   !str_detect(desc, negacion))
sum(str_detect(avisos$desc, negacion))
mean(avisos$cochera)

> 💡 **Lectura.** 1.002 avisos mencionan la cochera para negarla. Al descontarlos, la proporción de avisos con cochera baja de 34,2% a 31,9%.

### Los bordes de palabra

El SUM (salón de usos múltiples) es un amenity. Buscar `"sum"` sin más:

In [27]:
c(sin_borde = sum(str_detect(avisos$desc, "sum")),
  con_borde = sum(str_detect(avisos$desc, "\\bsum\\b")))

# Qué palabras contienen "sum" pero no son "sum"
palabras <- unlist(str_extract_all(avisos$desc, "\\w*sum\\w*"))
palabras[palabras != "sum"] |> table() |> sort(decreasing = TRUE) |> head(10)

sin_borde con_borde 
     9002      7280


                   consumidor                       consumo 
                         1235                           200 
                    sumamente                 suministrados 
                          180                           105 
                 suministrada                 suministradas 
                           74                            55 
                     sumaprop descubiertasolariumgymsumarea 
                           51                            46 
                         suma                    suministro 
                           44                            43 

> 💡 **Lectura.** Sin borde de palabra, "consumidor" (la leyenda de defensa del consumidor que traen muchos avisos), "sumamente" y "consumo" cuentan como SUM. `\\b` exige que el patrón empiece y termine en un borde de palabra. Fijate también "descubiertasolariumgymsumarea": avisos donde al sacar el HTML las palabras quedaron pegadas, y ahí ni el `\\b` las encuentra.

¿Cuánto más cuesta el metro cuadrado con cochera?

In [ ]:
avisos |>
  filter(!is.na(m2), between(m2, 20, 500)) |>
  group_by(cochera) |>
  summarise(avisos = n(), mediana_usd_m2 = median(precio / m2))

> 💡 **Lectura.** La mediana del metro cuadrado es de 3.478 dólares en los avisos con cochera y de 2.838 en los que no la mencionan: 23% más. Todavía no es el "precio de la cochera": lo retomamos en la Parte 4.

## 1.7 `str_extract()` y `str_match()`: sacar datos del texto

### Ambientes

`str_extract()` devuelve el texto que coincide; `str_match()` además separa lo que está entre paréntesis. La primera columna es la coincidencia completa y la segunda, el grupo:

In [32]:
str_match(c("hermoso 3 ambientes", "2amb c/balcon", "monoambiente", "vista abierta"),
          "(\\d{1,2}) ?amb")

3 amb,3
2amb,2
NA,NA
NA,NA


Del título, el número de ambientes. "Monoambiente" no tiene número: es 1.

In [29]:
avisos <- avisos |>
  mutate(amb_num  = as.numeric(str_match(tit, "\\b(\\d{1,2}) ?amb")[, 2]),
         amb_mono = str_detect(tit, "mono ?-?ambiente"),
         amb_tit  = case_when(!is.na(amb_num) ~ amb_num,
                              amb_mono ~ 1))
mean(!is.na(avisos$amb_tit))

[1] 0.5894854

In [31]:
avisos |>
  mutate(amb_num  = as.numeric(str_match(tit, "\\b(\\d{1,2}) ?amb")[, 2]),
         amb_mono = str_detect(tit, "mono ?-?ambiente"),
         amb_tit  = case_when(!is.na(amb_num) ~ amb_num,
                              amb_mono ~ 1)) |> 
select(tit,amb_num,amb_mono,amb_tit)

tit,amb_num,amb_mono,amb_tit
<chr>,<dbl>,<lgl>,<dbl>
juan j castelli 100 12° - u$d 68.000 - departamento en venta,NA,FALSE,NA
carlos calvo 3700 4° - u$d 130.215 - departamento en venta,NA,FALSE,NA
carlos calvo 3700 1° - u$d 120.108 - departamento en venta,NA,FALSE,NA
castro barros 1100 4° - u$d 110.000 - departamento en venta,NA,FALSE,NA
carlos calvo 3700 4° - u$d 92.153 - departamento en venta,NA,FALSE,NA
pavon 3300 3° - u$d 130.000 - departamento en venta,NA,FALSE,NA
carlos calvo 3700 2° - u$d 130.215 - departamento en venta,NA,FALSE,NA
marcelo t de alvear 400 3° - u$d 75.000 - departamento en venta,NA,FALSE,NA
colombres 100 1° - u$d 199.000 - departamento en venta,NA,FALSE,NA


**Validar**: donde el aviso trae los ambientes en la columna, ¿coincide con lo que sacamos del texto?

In [ ]:
avisos |>
  filter(!is.na(ambientes), !is.na(amb_tit)) |>
  summarise(comparados = n(), coincide = mean(ambientes == amb_tit))

avisos |> count(ambientes, amb_tit) |> filter(!is.na(ambientes), !is.na(amb_tit), ambientes != amb_tit) |>
  arrange(desc(n)) |> head(6)

> 💡 **Lectura.** El título trae los ambientes en el 59% de los avisos y, cuando también está la columna, coinciden en el 97% de los casos. La diferencia más común es 1 contra 2: un "2 ambientes" que en la columna figura como 1 (o al revés). Con esa tasa de acierto, el dato del texto sirve para **completar** los que faltan.

Si no está en el título, se busca en la descripción. Y en la descripción aparecen los números en palabras ("tres ambientes"): se convierten antes de buscar, con un reemplazo con grupo para no tocar el "dos" de "a dos cuadras".

In [ ]:
numeros <- c("\\bun (amb)" = "1 \\1", "\\bdos (amb)" = "2 \\1", "\\btres (amb)" = "3 \\1",
             "\\bcuatro (amb)" = "4 \\1", "\\bcinco (amb)" = "5 \\1")
str_replace_all("a dos cuadras del subte, tres ambientes", numeros)

avisos <- avisos |>
  mutate(desc_n    = str_replace_all(desc, numeros),
         amb_num   = as.numeric(str_match(desc_n, "\\b(\\d{1,2}) ?amb")[, 2]),
         amb_desc  = case_when(!is.na(amb_num) ~ amb_num,
                               str_detect(desc_n, "mono ?-?ambiente") ~ 1),
         amb_texto = coalesce(amb_tit, amb_desc))

avisos |>
  summarise(faltaban    = sum(is.na(ambientes)),
            recuperados = sum(is.na(ambientes) & !is.na(amb_texto)),
            coincide_desc = mean(ambientes == amb_desc, na.rm = TRUE))

> 💡 **Lectura.** De los 2.729 avisos sin ambientes en la columna, el texto permite recuperar casi todos. Pero la descripción es menos confiable que el título (coincide en el 89%): ahí aparecen frases como "edificio de 2 ambientes por piso". Por eso `coalesce()` usa primero el título y solo si falta, la descripción.

### Metros cuadrados

Para los metros conviene `str_extract_all()`, porque una descripción menciona varios ("56 m2 totales, living de 20 m2, balcón de 4 m2"). Pasos: unificar unidades, sacar el punto de miles (`1.200` → `1200`) y extraer **todos** los números seguidos de `m2`.

In [ ]:
d <- avisos$desc |>
  str_replace_all("(\\d)\\.(\\d{3})\\b", "\\1\\2") |>   # 1.200 -> 1200
  str_replace_all(unidades)

m2_todos <- str_extract_all(d, "\\d+(?:[.,]\\d+)? ?m2\\b") |>
  map(\(v) parse_number(v, locale = locale(decimal_mark = ",")))

# Las primeras tres descripciones que mencionan más de una superficie
head(keep(m2_todos, \(v) length(v) > 1), 3)

En el primero, 1.613 m2 no puede ser la superficie de un departamento que en el mismo texto tiene 28; el segundo lista nueve superficies, como hacen los avisos de emprendimientos con varias unidades. Cuál de todos usar es una **decisión**. Comparamos dos reglas contra la superficie que informa el aviso:

In [ ]:
avisos <- avisos |>
  mutate(m2_primero = map_dbl(m2_todos, \(v) if (length(v) == 0) NA else v[1]),
         m2_maximo  = map_dbl(m2_todos, \(v) if (length(v) == 0) NA else max(v)))

avisos |>
  filter(!is.na(m2_total), !is.na(m2_primero)) |>
  summarise(comparados        = n(),
            primero_coincide = mean(abs(m2_primero - m2_total) <= 1),
            maximo_coincide  = mean(abs(m2_maximo - m2_total) <= 1))

> 💡 **Lectura.** Solo el 31% de las descripciones menciona metros con unidad. Cuando lo hace, el **máximo** coincide con la superficie total informada en el 66% de los casos y el primero, en el 63%: el primer número suele ser de un ambiente ("living de 20 m2"). Ninguna regla es perfecta; lo importante es haberlo medido.

> ⚠️ **Atención.** El patrón exige la unidad `m2`. Si se aceptara también "mts" o "metros" a secas, se colarían las distancias: "a 300 mts del subte" no es la superficie del departamento.

### Expensas: un grupo y `parse_number()`

"expensas", hasta 20 caracteres que no sean dígitos ni `$`, el signo pesos y el número:

In [ ]:
avisos <- avisos |>
  mutate(expensas_txt = str_match(desc, "expensas[^0-9$]{0,20}\\$ ?(\\d[\\d.]*)")[, 2],
         expensas     = parse_number(expensas_txt, locale = locale(grouping_mark = ".")))

mean(!is.na(avisos$expensas))
quantile(avisos$expensas, c(0, 0.01, 0.25, 0.5, 0.75, 0.99, 1), na.rm = TRUE)

Siempre mirar los extremos:

In [ ]:
avisos |> slice_max(expensas, n = 4) |>
  transmute(expensas, contexto = str_extract(desc, "expensas.{0,50}"))

> 💡 **Lectura.** El 16% de los avisos informa expensas; la mediana es $3.400 (precios de 2018-2019). Los máximos no son expensas: son las expensas y el precio pegados ("$8700" y "214.000 dólares" quedaron escritos como "$8700214.000") y el patrón los leyó como un solo número. Antes de usar la variable hay que fijar un rango razonable y dejar el resto como `NA`.

In [ ]:
avisos <- avisos |>
  mutate(expensas = if_else(between(expensas, 300, 60000), expensas, NA_real_))
summary(avisos$expensas)

## 1.8 Cómo hacerlo bien

| Regla | Por qué |
|:--|:--|
| No pisar la columna original | Para poder volver a mirar el texto cuando algo da raro |
| Normalizar primero (HTML, minúsculas, tildes, espacios), buscar después | Un patrón sobre texto normalizado es más corto y falla menos |
| Probar el patrón con `str_view()` en ejemplos chicos | Se ve exactamente qué coincide |
| Mirar el contexto con `".{0,25}patron.{0,25}"` | Aparecen las negaciones ("sin cochera") y los falsos positivos ("consumidor") |
| Usar `\\b` para palabras cortas | `"sum"` encuentra "consumo"; `"\\bsum\\b"`, no |
| Contar: cuántos encontró, cuántos `NA` quedaron | Una variable extraída del 5% de los avisos no sirve para mucho |
| Validar contra un dato conocido | Los ambientes del título coinciden en el 97%; los m2 de la descripción, en el 66% |
| Mirar los extremos y fijar rangos razonables | Las expensas de 8.700 millones eran números pegados |
| Para categorías, reglas con `case_when()` en orden | Más fácil de leer y de corregir que una regex gigante |
| No usar regex para lo que ya tiene función | `parse_number()`, `str_to_title()`, `str_squish()`, `lubridate` para fechas |

# Parte 2. Funciones propias

## 2.1 Cuándo escribir una función

En 1.5 escribimos **dos veces** la misma limpieza, una para la descripción y otra para el título: copiar, pegar y cambiar el nombre de la columna. Si mañana hay que agregar un paso (por ejemplo, sacar los teléfonos), hay que acordarse de cambiarlo en todos lados.

La regla práctica: **si copiaste y pegaste el mismo bloque tres veces, escribí una función.** Ventajas:

- El cambio se hace en un solo lugar.
- El nombre dice qué hace (`limpiar_texto()` se entiende mejor que cinco líneas de regex).
- Se puede probar con ejemplos chicos.

## 2.2 Anatomía de una función

```r
nombre <- function(argumento1, argumento2 = valor_por_defecto) {
  # cuerpo: lo que se hace con los argumentos
  resultado    # la última línea es lo que devuelve
}
```

La limpieza de 1.5, como función:

In [ ]:
limpiar_texto <- function(x) {
  x |>
    str_replace_all("<[^>]+>", " ") |>
    str_replace_all("&[a-z]+;", " ") |>
    str_to_lower() |>
    stringi::stri_trans_general("Latin-ASCII") |>
    str_squish()
}

limpiar_texto(c("  <p>Hermoso 2 AMB.</p>&nbsp;Balcón ", "MONO-AMBIENTE en Núñez"))

Da exactamente lo mismo que el código copiado y pegado:

In [ ]:
identical(limpiar_texto(avisos$titulo), avisos$tit)

## 2.3 Argumentos con valor por defecto

A veces las tildes sí importan (por ejemplo, para mostrar los nombres en un gráfico). Un argumento con valor por defecto permite elegir sin complicar el uso habitual: si no se lo escribe, vale `TRUE`.

In [ ]:
limpiar_texto <- function(x, sacar_tildes = TRUE) {
  x <- x |>
    str_replace_all("<[^>]+>", " ") |>
    str_replace_all("&[a-z]+;", " ") |>
    str_to_lower()
  if (sacar_tildes) x <- stringi::stri_trans_general(x, "Latin-ASCII")
  str_squish(x)
}

limpiar_texto("Balcón   en NÚÑEZ")
limpiar_texto("Balcón   en NÚÑEZ", sacar_tildes = FALSE)

## 2.4 Funciones que sacan datos del texto

Lo que en 1.7 fueron varias líneas sueltas, ahora con nombre. Una función **hace una sola cosa** y recibe todo lo que necesita como argumento.

In [ ]:
# Número de ambientes en un texto ya limpio. "Monoambiente" cuenta como 1.
extraer_ambientes <- function(texto) {
  texto <- str_replace_all(texto, c("\\bun (amb)" = "1 \\1", "\\bdos (amb)" = "2 \\1",
                                    "\\btres (amb)" = "3 \\1", "\\bcuatro (amb)" = "4 \\1",
                                    "\\bcinco (amb)" = "5 \\1"))
  numero <- as.numeric(str_match(texto, "\\b(\\d{1,2}) ?amb")[, 2])
  case_when(!is.na(numero) ~ numero,
            str_detect(texto, "mono ?-?ambiente") ~ 1)
}

# Superficie en m2 en un texto ya limpio. Si hay varias, elige según `criterio`.
extraer_m2 <- function(texto, criterio = c("maximo", "primero")) {
  criterio <- match.arg(criterio)
  texto <- texto |>
    str_replace_all("(\\d)\\.(\\d{3})\\b", "\\1\\2") |>
    str_replace_all(c("m²" = "m2", "mts2" = "m2", "mt2" = "m2", "metros cuadrados" = "m2"))
  todos <- str_extract_all(texto, "\\d+(?:[.,]\\d+)? ?m2\\b") |>
    map(\(v) parse_number(v, locale = locale(decimal_mark = ",")))
  elegir <- if (criterio == "maximo") max else \(v) v[1]
  map_dbl(todos, \(v) if (length(v) == 0) NA else elegir(v))
}

extraer_ambientes(c("hermoso tres ambientes", "2amb c/balcon", "monoambiente", "a dos cuadras"))
extraer_m2(c("56 m2 totales, living de 20 m2", "1.200 mts2", "sin datos"))
extraer_m2(c("56 m2 totales, living de 20 m2", "1.200 mts2", "sin datos"), criterio = "primero")

`match.arg()` hace dos cosas: si no se elige `criterio`, toma la primera opción (`"maximo"`), y si se escribe algo que no está en la lista, da un error claro.

Ahora toda la Parte 1 entra en un `mutate()` que se lee de corrido:

In [ ]:
avisos <- avisos |>
  mutate(desc      = limpiar_texto(descripcion),
         tit       = limpiar_texto(titulo),
         amb_texto = coalesce(extraer_ambientes(tit), extraer_ambientes(desc)),
         m2_texto  = extraer_m2(desc))

avisos |> summarise(across(c(amb_texto, m2_texto), \(v) mean(!is.na(v))))

## 2.5 Validar lo que entra

Si alguien le pasa un número a `extraer_m2()`, la función devuelve `NA` **sin avisar**, que es peor que un error: el problema aparece mucho después, como una columna vacía. Con `stop()` se corta con un mensaje claro. `try()` muestra el error sin cortar el notebook:

In [ ]:
try(extraer_m2(56))

extraer_m2 <- function(texto, criterio = c("maximo", "primero")) {
  if (!is.character(texto)) stop("`texto` tiene que ser texto, no ", class(texto)[1])
  criterio <- match.arg(criterio)
  texto <- texto |>
    str_replace_all("(\\d)\\.(\\d{3})\\b", "\\1\\2") |>
    str_replace_all(c("m²" = "m2", "mts2" = "m2", "mt2" = "m2", "metros cuadrados" = "m2"))
  todos <- str_extract_all(texto, "\\d+(?:[.,]\\d+)? ?m2\\b") |>
    map(\(v) parse_number(v, locale = locale(decimal_mark = ",")))
  elegir <- if (criterio == "maximo") max else \(v) v[1]
  map_dbl(todos, \(v) if (length(v) == 0) NA else elegir(v))
}

try(extraer_m2(56))
try(extraer_m2("56 m2", criterio = "mediana"))

## 2.6 Funciones que reciben columnas de un data frame

Para repetir el mismo resumen agrupando por distintas variables, la columna se pasa como argumento y se escribe entre **llaves dobles** `{{ }}` dentro de las funciones de `dplyr`. Sin las llaves, `group_by()` buscaría una columna que se llame literalmente `grupo`.

In [ ]:
resumen_usd_m2 <- function(datos, grupo) {
  datos |>
    filter(!is.na(m2), between(m2, 20, 500)) |>
    group_by({{ grupo }}) |>
    summarise(avisos = n(), mediana_usd_m2 = median(precio / m2), .groups = "drop")
}

resumen_usd_m2(avisos, cochera)
resumen_usd_m2(avisos, amb_texto) |> filter(avisos >= 100)

> ⚠️ **Atención: lo que pasa adentro queda adentro.** Las variables creadas dentro de una función (como `todos` o `elegir` en `extraer_m2()`) no existen afuera. Y al revés: una función no debería usar objetos que no recibió como argumento. Si `resumen_usd_m2()` usara `avisos` en vez de `datos`, funcionaría hoy pero daría resultados equivocados el día que se la llame con otra base.

# Parte 3. Loops y condicionales

## 3.1 `if` / `else`: decidir qué hacer

`if` evalúa **una** condición y ejecuta un bloque si es `TRUE`; `else`, si es `FALSE`.

In [ ]:
tamanio <- function(m2) {
  if (m2 < 40) {
    "chico"
  } else if (m2 < 80) {
    "mediano"
  } else {
    "grande"
  }
}
tamanio(35)
tamanio(120)

In [ ]:
try(tamanio(c(35, 120)))

> ⚠️ **Atención.** `if` es para **un solo valor**: decide qué código correr. Para clasificar una columna entera (un valor por fila), se usa `if_else()` o `case_when()`, que trabajan con vectores. Es el error más común al empezar: `if (avisos$m2 < 40)` da error porque `avisos$m2` tiene 44 mil valores.

In [ ]:
avisos |>
  mutate(tamanio = case_when(m2 < 40 ~ "chico", m2 < 80 ~ "mediano", m2 >= 80 ~ "grande")) |>
  count(tamanio)

## 3.2 `for`: repetir para cada elemento

`for (x in vector)` corre el bloque una vez por cada elemento, y en cada vuelta `x` vale uno distinto.

In [ ]:
for (b in c("PALERMO", "FLORES", "VILLA LUGANO")) {
  sub <- filter(avisos, barrio == b, between(m2, 20, 500))
  cat(b, ": mediana de", round(median(sub$precio / sub$m2)), "dólares por m2\n")
}

## 3.3 Guardar lo que sale del loop

Lo que se imprime con `cat()` se pierde. Para **guardar**, se crea una lista vacía antes del loop y en cada vuelta se agrega un elemento; al final, `bind_rows()` la convierte en una tabla.

El caso típico: crear muchas dummies a partir de una lista de patrones. Sin loop serían diez líneas casi iguales dentro de un `mutate()`.

In [ ]:
amenities <- c(
  pileta     = "pileta|piscina",
  gimnasio   = "gimnasio|\\bgym\\b",
  sum        = "\\bsum\\b|salon de usos multiples",
  parrilla   = "parrilla",
  laundry    = "laundry|lavadero comun",
  seguridad  = "seguridad 24|vigilancia",
  balcon     = "balcon",
  terraza    = "terraza",
  a_estrenar = "a estrenar"
)

resultados <- list()

for (nombre in names(amenities)) {
  avisos[[nombre]] <- str_detect(avisos$desc, amenities[[nombre]])

  con <- avisos |> filter(.data[[nombre]], between(m2, 20, 500))
  sin <- avisos |> filter(!.data[[nombre]], between(m2, 20, 500))

  resultados[[nombre]] <- tibble(
    amenity         = nombre,
    porcentaje      = round(100 * mean(avisos[[nombre]]), 1),
    usd_m2_con      = median(con$precio / con$m2),
    usd_m2_sin      = median(sin$precio / sin$m2),
    diferencia_pct  = round(100 * (usd_m2_con / usd_m2_sin - 1), 1)
  )
}

tabla_amenities <- bind_rows(resultados) |> arrange(desc(diferencia_pct))
tabla_amenities

> 💡 **Lectura.** Un loop de diez líneas reemplazó nueve dummies y nueve resúmenes. El metro cuadrado de los avisos que mencionan gimnasio, pileta o seguridad es bastante más caro que el del resto, pero eso **no** es el precio del gimnasio: los edificios con amenities están en otros barrios y son más nuevos. Para separar eso hace falta una regresión (Parte 4).

Dos detalles del código:

- `avisos[[nombre]]` crea o lee una columna cuyo nombre está guardado en una variable. `avisos$nombre` buscaría una columna que se llame "nombre".
- Dentro de `filter()`, lo mismo se escribe `.data[[nombre]]`.

## 3.4 Condicionales dentro del loop: `next` y avisos

Un loop por barrio: en los barrios con pocos avisos la mediana es poco confiable, así que **se saltean** con `next` (que pasa directo a la vuelta siguiente). Además, se marca cuando a un barrio le faltan muchos metros cuadrados.

In [ ]:
barrios <- sort(unique(avisos$barrio))
por_barrio <- list()

for (b in barrios) {
  sub <- filter(avisos, barrio == b)

  if (nrow(sub) < 150) {
    message("Salteo ", b, ": solo ", nrow(sub), " avisos")
    next
  }

  falta_m2 <- mean(is.na(sub$m2))
  calidad  <- if (falta_m2 > 0.05) "revisar" else "ok"

  sub <- filter(sub, between(m2, 20, 500))
  por_barrio[[b]] <- tibble(barrio = b, avisos = nrow(sub),
                            usd_m2 = median(sub$precio / sub$m2),
                            falta_m2 = round(100 * falta_m2, 1), calidad = calidad)
}

por_barrio <- bind_rows(por_barrio) |> arrange(desc(usd_m2))
nrow(por_barrio)
head(por_barrio, 5)
filter(por_barrio, calidad == "revisar")

> 💡 **Lectura.** De 48 barrios, 7 se saltearon por tener menos de 150 avisos y quedan 41. Puerto Madero es el más caro (6.316 dólares por m2), pero al 10% de sus avisos le faltan los metros: queda marcado para revisar antes de usar ese número.

| Dentro de un loop | Qué hace |
|:--|:--|
| `if (...) { }` | Corre un bloque solo si se cumple la condición |
| `next` | Saltea lo que queda de esta vuelta y pasa a la siguiente |
| `break` | Corta el loop entero |
| `message()`, `warning()` | Avisan sin cortar (`stop()` sí corta) |

## 3.5 El uso más común en un TP: leer muchos archivos

Muchas fuentes vienen en un archivo por mes, por año o por provincia. Simulamos el caso: guardamos un CSV por mes y después los leemos con un loop.

In [ ]:
dir.create(file.path(carpeta_datos, "por_mes"), showWarnings = FALSE)

avisos |>
  mutate(mes = format(fecha, "%Y_%m")) |>
  select(mes, barrio, ambientes, m2, precio) |>
  group_split(mes) |>
  walk(\(d) write_csv(select(d, -mes), file.path(carpeta_datos, "por_mes", paste0("avisos_", d$mes[1], ".csv"))))

archivos <- list.files(file.path(carpeta_datos, "por_mes"), pattern = "\\.csv$", full.names = TRUE)
basename(archivos)

In [ ]:
lista <- list()

for (archivo in archivos) {
  d <- read_csv(archivo, show_col_types = FALSE)
  if (nrow(d) == 0) next
  d$mes <- str_extract(basename(archivo), "\\d{4}_\\d{2}")    # el mes sale del nombre del archivo
  lista[[archivo]] <- d
}

todos <- bind_rows(lista)
count(todos, mes)

> 💡 **Lectura.** El nombre del archivo es un dato: `str_extract()` con `"\\d{4}_\\d{2}"` saca el año y el mes de "avisos_2019_02.csv" y queda como columna. Sin eso, al juntar los archivos se pierde de qué mes es cada fila.

## 3.6 ¿Loop, vector o `map()`?

En R muchas operaciones ya trabajan con vectores enteros y no necesitan loop. La misma cuenta de tres formas:

In [ ]:
# 1. Loop
largo <- numeric(length(avisos$desc))     # el vector se crea antes, con el largo final
for (i in seq_along(avisos$desc)) {
  largo[i] <- str_length(avisos$desc[i])
}

# 2. Vectorizado: la función ya recorre el vector
largo_v <- str_length(avisos$desc)

# 3. map(): aplica una función a cada elemento y junta los resultados
largo_m <- map_dbl(avisos$desc, str_length)

c(all(largo == largo_v), all(largo == largo_m))

Y la tabla de amenities de 3.3, con `map()`:

In [ ]:
map_dfr(names(amenities), \(nombre) {
  avisos |>
    filter(between(m2, 20, 500)) |>
    group_by(tiene = .data[[nombre]]) |>
    summarise(usd_m2 = median(precio / m2)) |>
    pivot_wider(names_from = tiene, values_from = usd_m2, names_prefix = "usd_m2_") |>
    mutate(amenity = nombre, .before = 1)
})

| Cuándo | Qué usar |
|:--|:--|
| La función ya acepta vectores (`str_*`, aritmética, `if_else`) | Vectorizado, sin loop |
| Aplicar lo mismo a cada elemento y juntar los resultados | `map()`, `map_dbl()`, `map_dfr()` |
| Cada vuelta depende de la anterior, hay `next`/`break`, o se leen y escriben archivos | `for` |
| El código del loop es largo | Escribir una función y llamarla desde el loop o desde `map()` |

> ⚠️ **Atención.** Si se usa un `for`, crear el objeto **antes** con el largo final (`numeric(n)`, `list()`) y llenarlo. Hacer crecer un vector con `x <- c(x, nuevo)` en cada vuelta funciona, pero con muchas vueltas se vuelve muy lento.

# Parte 4. Integrador: del texto a la regresión

Todo junto: limpiar con funciones, crear dummies con un loop y estimar el precio. Los ambientes que faltan se completan con los del texto; nos quedamos con superficies y precios razonables y con los barrios con al menos 150 avisos.

In [ ]:
base <- avisos |>
  mutate(ambientes = coalesce(ambientes, amb_texto)) |>
  filter(between(m2, 20, 500), between(precio, 20000, 3e6), ambientes <= 8,
         barrio %in% por_barrio$barrio)

r1 <- lm(log(precio) ~ log(m2) + ambientes + barrio, data = base)
r2 <- update(r1, . ~ . + cochera + pileta + gimnasio + seguridad + balcon + a_estrenar)

mostrar_tabla(list("Sin texto" = r1, "Con variables del texto" = r2),
  vcov     = "HC1",
  coef_omit = "barrio|Intercept",
  gof_map  = c("nobs", "r.squared"),
  stars    = TRUE,
  notas    = c("Variable dependiente: log(precio en dólares). Incluye efectos fijos por barrio.",
               "Errores robustos (HC1) entre paréntesis. + p < 0,1; * p < 0,05; ** p < 0,01; *** p < 0,001"))

In [ ]:
round(100 * (exp(coef(r2)[c("cocheraTRUE", "piletaTRUE", "seguridadTRUE", "a_estrenarTRUE")]) - 1), 1)

> 💡 **Lectura.** A igual superficie, ambientes y barrio, un departamento cuyo aviso menciona cochera cuesta alrededor de 14% más; con pileta, 11% más, y a estrenar, 9% más. Las variables que salieron del texto suben el $R^2$ de 0,865 a 0,893. Son variables construidas con reglas imperfectas (validamos que los ambientes coinciden en el 97%, pero la cochera puede estar mal marcada en algunos avisos): ese error de medición tiende a **achicar** los coeficientes hacia cero.

# Cierre

| Herramienta | Para qué | Cuidado |
|:--|:--|:--|
| `str_squish()`, `str_to_lower()`, `stri_trans_general()` | Normalizar antes de comparar | No pisar la columna original |
| `str_replace_all()` | Limpiar (HTML, unidades, números en palabras) | Probar el patrón con `str_view()` |
| `str_detect()` | Crear dummies y reglas con `case_when()` | Negaciones y bordes de palabra (`\\b`) |
| `str_extract()`, `str_match()` | Sacar números y categorías del texto | Validar contra un dato conocido y mirar extremos |
| `function()` | No repetir código; nombrar lo que se hace | Una tarea por función; todo lo que usa entra como argumento |
| `if` / `else` | Decidir qué código correr (un valor) | Para columnas, `if_else()` o `case_when()` |
| `for`, `next`, `break` | Repetir con decisiones en el medio, leer archivos | Crear antes el objeto donde se guarda |
| `map()`, `map_dfr()` | Lo mismo que un `for` simple, más corto | |